# Shared Tokenization Pipeline — TweetEval Sentiment

Notebook dùng chung cho cả nhóm.

## Mục tiêu
- Fit vocabulary **chỉ trên train**
- Dùng chung tokenizer cho train / validation / test
- Chuẩn hóa tweet nhưng giữ tín hiệu sentiment
- Xuất sẵn `X_train`, `X_validation`, `X_test`, `y_*`
- Lưu vocabulary + metadata để cả 3 model dùng giống nhau

> Sau khi notebook này chạy xong, người train model chỉ cần load các file `.npy` trong `artifacts/`.


## Quy tắc bắt buộc

1. Không fit/adapt tokenizer lại ở notebook model.
2. Không tự chia lại dataset.
3. Không đổi sequence length.
4. Không dùng test set để tune tokenizer hoặc hyperparameter.
5. Các model chỉ nên bắt đầu khác nhau từ phần **Embedding / kiến trúc model**.


In [ ]:
# Cài dependency (chạy 1 lần)
!pip -q install emoji


In [ ]:
import os
import re
import html
import json
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import emoji

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 1. Cấu hình

Nếu tên cột text/label khác mặc định, sửa `TEXT_COLUMN` và `LABEL_COLUMN`.

Nếu để `None`, notebook sẽ thử tự phát hiện.


In [ ]:
DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

TRAIN_FILE = "train.csv"
VALIDATION_FILE = "validation.csv"
TEST_FILE = "test.csv"

TEXT_COLUMN = None
LABEL_COLUMN = None

MAX_TOKENS = 30000

# sequence length sẽ được chọn tự động theo p95 của TRAIN
SEQ_PERCENTILE = 95
SEQ_MIN = 32
SEQ_MAX = 96

LOWERCASE = True
REPLACE_URLS = True
REPLACE_MENTIONS = True
HASHTAG_MODE = "marker_and_text"   # marker_and_text | text_only | keep | remove
DEMOJIZE_EMOJIS = True
PRESERVE_EMPHASIS = True


## 2. Đặt dataset

Cấu trúc mong muốn:

```text
data/
├── train.csv
├── validation.csv
└── test.csv
```

### Nếu dùng Google Colab
Bạn có thể upload 3 file CSV trực tiếp:


In [ ]:
# CHỈ CHẠY CELL NÀY NẾU DÙNG GOOGLE COLAB VÀ MUỐN UPLOAD FILE
# from google.colab import files
# uploaded = files.upload()
#
# DATA_DIR.mkdir(exist_ok=True)
# for name, content in uploaded.items():
#     if name in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]:
#         (DATA_DIR / name).write_bytes(content)
#
# print("Đã copy file vào:", DATA_DIR.resolve())


In [ ]:
TEXT_CANDIDATES = ["text", "tweet", "sentence", "content", "review", "message", "comment"]
LABEL_CANDIDATES = ["label", "sentiment", "target", "class", "category"]

def detect_column(df, explicit, candidates, kind):
    if explicit is not None:
        if explicit not in df.columns:
            raise ValueError(f"Không tìm thấy cột {kind}='{explicit}'. Có: {list(df.columns)}")
        return explicit

    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c in lower_map:
            return lower_map[c]

    raise ValueError(
        f"Không tự phát hiện được cột {kind}. "
        f"Hãy đặt biến {kind.upper()}_COLUMN thủ công."
    )

train_df = pd.read_csv(DATA_DIR / TRAIN_FILE)
val_df = pd.read_csv(DATA_DIR / VALIDATION_FILE)
test_df = pd.read_csv(DATA_DIR / TEST_FILE)

text_col = detect_column(train_df, TEXT_COLUMN, TEXT_CANDIDATES, "text")
label_col = detect_column(train_df, LABEL_COLUMN, LABEL_CANDIDATES, "label")

print("Text column :", text_col)
print("Label column:", label_col)
print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)


## 3. Chuẩn hóa tweet

Pipeline giữ lại các tín hiệu sentiment quan trọng:

- URL → `<url>`
- mention → `<user>`
- hashtag → `<hashtag>` + nội dung
- emoji → token mô tả
- `!`, `!!`, `!!!` → token nhấn mạnh khác nhau
- `?`, `??`, `???` → token khác nhau
- giữ contraction như `don't`


In [ ]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"(?<!\w)@[A-Za-z0-9_]+")
HASHTAG_RE = re.compile(r"#([A-Za-z0-9_]+)")

def split_hashtag(tag):
    tag = tag.replace("_", " ")
    tag = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", tag)
    return re.sub(r"\s+", " ", tag).strip()

def replace_hashtag(match):
    content = split_hashtag(match.group(1))
    if HASHTAG_MODE == "keep":
        return f" #{content} "
    if HASHTAG_MODE == "text_only":
        return f" {content} "
    if HASHTAG_MODE == "marker_and_text":
        return f" <hashtag> {content} "
    if HASHTAG_MODE == "remove":
        return " "
    raise ValueError("HASHTAG_MODE không hợp lệ")

def normalize_tweet(text):
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)

    if REPLACE_URLS:
        text = URL_RE.sub(" <url> ", text)

    if REPLACE_MENTIONS:
        text = MENTION_RE.sub(" <user> ", text)

    text = HASHTAG_RE.sub(replace_hashtag, text)

    if DEMOJIZE_EMOJIS:
        text = emoji.demojize(text, delimiters=(" emoji_", " "))
        text = re.sub(
            r"emoji_([A-Za-z0-9_+\-]+)",
            lambda m: "emoji_" + m.group(1).replace("-", "_"),
            text,
        )

    if PRESERVE_EMPHASIS:
        text = re.sub(r"!{3,}", " <exclaim3> ", text)
        text = re.sub(r"!{2}", " <exclaim2> ", text)
        text = re.sub(r"!", " <exclaim> ", text)

        text = re.sub(r"\?{3,}", " <question3> ", text)
        text = re.sub(r"\?{2}", " <question2> ", text)
        text = re.sub(r"\?", " <question> ", text)

        text = re.sub(r"\.{3,}", " <ellipsis> ", text)

    text = re.sub(r"[^\w\s'<>\-]", " ", text, flags=re.UNICODE)
    text = text.replace("-", " ")

    if LOWERCASE:
        text = text.lower()

    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [train_df, val_df, test_df]:
    if df[text_col].isna().any():
        raise ValueError("Có text null. Không tự drop để tránh lệch split.")
    if df[label_col].isna().any():
        raise ValueError("Có label null. Không tự drop để tránh lệch split.")

train_norm = train_df[text_col].map(normalize_tweet).tolist()
val_norm = val_df[text_col].map(normalize_tweet).tolist()
test_norm = test_df[text_col].map(normalize_tweet).tolist()

print("Ví dụ trước:")
print(train_df[text_col].iloc[0])
print("\nSau:")
print(train_norm[0])


## 4. Chọn sequence length chỉ từ TRAIN


In [ ]:
train_lengths = np.array([len(x.split()) for x in train_norm], dtype=np.int32)

raw_seq_len = int(np.ceil(np.percentile(train_lengths, SEQ_PERCENTILE)))
SEQUENCE_LENGTH = max(SEQ_MIN, min(SEQ_MAX, raw_seq_len))

print("p50:", np.percentile(train_lengths, 50))
print("p90:", np.percentile(train_lengths, 90))
print("p95:", np.percentile(train_lengths, 95))
print("p99:", np.percentile(train_lengths, 99))
print("max:", train_lengths.max())
print("Chosen sequence length:", SEQUENCE_LENGTH)
print("Estimated train truncation rate:", np.mean(train_lengths > SEQUENCE_LENGTH))


## 5. Fit vocabulary chỉ trên TRAIN


In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize=None,
    split="whitespace",
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)

train_text_ds = tf.data.Dataset.from_tensor_slices(train_norm).batch(1024)
vectorizer.adapt(train_text_ds)

vocab = vectorizer.get_vocabulary()

print("Actual vocabulary size:", len(vocab))
print("Reserved token IDs:")
print("0 -> padding/mask:", repr(vocab[0]))
print("1 -> OOV:", repr(vocab[1]))


## 6. Transform train / validation / test bằng đúng tokenizer này


In [ ]:
def vectorize_in_batches(texts, batch_size=1024):
    ds = tf.data.Dataset.from_tensor_slices(texts).batch(batch_size)
    chunks = [vectorizer(batch).numpy().astype(np.int32) for batch in ds]
    return np.concatenate(chunks, axis=0) if chunks else np.empty((0, SEQUENCE_LENGTH), dtype=np.int32)

X_train = vectorize_in_batches(train_norm)
X_val = vectorize_in_batches(val_norm)
X_test = vectorize_in_batches(test_norm)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


## 7. Label mapping


In [ ]:
def build_label_mapping(series):
    vals = list(pd.unique(series))

    numeric_vals = []
    numeric = True
    for v in vals:
        try:
            fv = float(v)
            if not fv.is_integer():
                numeric = False
                break
            numeric_vals.append(int(fv))
        except:
            numeric = False
            break

    if numeric and set(numeric_vals) == {0, 1, 2}:
        return {"0": 0, "1": 1, "2": 2}

    normalized = {str(v).strip().lower(): v for v in vals}
    if all(x in normalized for x in ["negative", "neutral", "positive"]):
        return {
            str(normalized["negative"]): 0,
            str(normalized["neutral"]): 1,
            str(normalized["positive"]): 2,
        }

    sorted_vals = sorted([str(v) for v in vals])
    return {v: i for i, v in enumerate(sorted_vals)}

def encode_labels(series, mapping):
    out = []
    for v in series:
        key = str(v)
        if key not in mapping:
            try:
                fv = float(v)
                if fv.is_integer() and str(int(fv)) in mapping:
                    key = str(int(fv))
            except:
                pass
        if key not in mapping:
            raise ValueError(f"Label không có trong mapping: {v}")
        out.append(mapping[key])
    return np.asarray(out, dtype=np.int32)

label_mapping = build_label_mapping(train_df[label_col])

y_train = encode_labels(train_df[label_col], label_mapping)
y_val = encode_labels(val_df[label_col], label_mapping)
y_test = encode_labels(test_df[label_col], label_mapping)

print("Label mapping:", label_mapping)


## 8. Kiểm tra OOV và truncation


In [ ]:
def oov_rate(x):
    non_pad = x != 0
    denom = int(non_pad.sum())
    if denom == 0:
        return 0.0
    return float(((x == 1) & non_pad).sum() / denom)

def truncation_rate(texts):
    lengths = np.asarray([len(x.split()) for x in texts])
    return float(np.mean(lengths > SEQUENCE_LENGTH))

print("OOV train      :", oov_rate(X_train))
print("OOV validation :", oov_rate(X_val))
print("OOV test       :", oov_rate(X_test))

print("Trunc train      :", truncation_rate(train_norm))
print("Trunc validation :", truncation_rate(val_norm))
print("Trunc test       :", truncation_rate(test_norm))


## 9. Lưu artifacts dùng chung cho cả team


In [ ]:
np.save(ARTIFACT_DIR / "X_train.npy", X_train)
np.save(ARTIFACT_DIR / "X_validation.npy", X_val)
np.save(ARTIFACT_DIR / "X_test.npy", X_test)

np.save(ARTIFACT_DIR / "y_train.npy", y_train)
np.save(ARTIFACT_DIR / "y_validation.npy", y_val)
np.save(ARTIFACT_DIR / "y_test.npy", y_test)

(ARTIFACT_DIR / "vocabulary.json").write_text(
    json.dumps(vocab, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

(ARTIFACT_DIR / "label_mapping.json").write_text(
    json.dumps(label_mapping, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

metadata = {
    "pipeline_version": "1.0-ipynb",
    "fit_policy": "Vocabulary adapted on TRAIN ONLY. Validation/Test transformed only.",
    "text_column": str(text_col),
    "label_column": str(label_col),
    "max_tokens": int(MAX_TOKENS),
    "actual_vocabulary_size": int(len(vocab)),
    "sequence_length": int(SEQUENCE_LENGTH),
    "padding_token_id": 0,
    "oov_token_id": 1,
    "label_mapping": label_mapping,
    "split_sizes": {
        "train": int(len(train_df)),
        "validation": int(len(val_df)),
        "test": int(len(test_df)),
    },
    "oov_rate": {
        "train": oov_rate(X_train),
        "validation": oov_rate(X_val),
        "test": oov_rate(X_test),
    },
    "truncation_rate": {
        "train": truncation_rate(train_norm),
        "validation": truncation_rate(val_norm),
        "test": truncation_rate(test_norm),
    },
    "normalization": {
        "lowercase": LOWERCASE,
        "replace_urls": REPLACE_URLS,
        "replace_mentions": REPLACE_MENTIONS,
        "hashtag_mode": HASHTAG_MODE,
        "demojize_emojis": DEMOJIZE_EMOJIS,
        "preserve_emphasis": PRESERVE_EMPHASIS,
    }
}

(ARTIFACT_DIR / "tokenizer_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Audit
pd.DataFrame({
    "text_original": train_df[text_col].astype(str),
    "text_normalized": train_norm,
    "label_original": train_df[label_col]
}).to_csv(ARTIFACT_DIR / "train_normalized_audit.csv", index=False)

pd.DataFrame({
    "text_original": val_df[text_col].astype(str),
    "text_normalized": val_norm,
    "label_original": val_df[label_col]
}).to_csv(ARTIFACT_DIR / "validation_normalized_audit.csv", index=False)

pd.DataFrame({
    "text_original": test_df[text_col].astype(str),
    "text_normalized": test_norm,
    "label_original": test_df[label_col]
}).to_csv(ARTIFACT_DIR / "test_normalized_audit.csv", index=False)

print("Đã lưu artifacts tại:", ARTIFACT_DIR.resolve())


## 10. Code mà người train model dùng

Sau khi nhận folder `artifacts/`, notebook model chỉ cần:


In [ ]:
# COPY đoạn này sang notebook model

import json
import numpy as np
from pathlib import Path

ARTIFACT_DIR = Path("artifacts")

X_train = np.load(ARTIFACT_DIR / "X_train.npy")
y_train = np.load(ARTIFACT_DIR / "y_train.npy")

X_val = np.load(ARTIFACT_DIR / "X_validation.npy")
y_val = np.load(ARTIFACT_DIR / "y_validation.npy")

X_test = np.load(ARTIFACT_DIR / "X_test.npy")
y_test = np.load(ARTIFACT_DIR / "y_test.npy")

meta = json.loads(
    (ARTIFACT_DIR / "tokenizer_metadata.json").read_text(encoding="utf-8")
)

VOCAB_SIZE = meta["actual_vocabulary_size"]
SEQUENCE_LENGTH = meta["sequence_length"]

print(X_train.shape, X_val.shape, X_test.shape)
print("VOCAB_SIZE =", VOCAB_SIZE)
print("SEQUENCE_LENGTH =", SEQUENCE_LENGTH)


## 11. Điểm bắt đầu của model

Người train model **không cần tokenize nữa**.

```text
X_train / X_val / X_test
        ↓
Embedding
        ↓
Transformer / BiLSTM / kiến trúc riêng
        ↓
Classifier
```


In [ ]:
# Ví dụ input model

inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32)

x = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=128,
    mask_zero=True
)(inputs)

print("Input shape:", inputs.shape)
print("Embedding output:", x.shape)

# Dừng ở đây.
# Mỗi người thay phần tiếp theo bằng kiến trúc model của mình.
